In [1]:
import subprocess, sys, os
import torch

print(f"Python : {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.version.cuda}")
print(f"GPUs   : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | "
          f"{torch.cuda.get_device_properties(i).total_memory // 1024**2} MB")

if torch.cuda.device_count() < 2:
    raise RuntimeError("Need 2 GPUs. Enable GPU T4 x2 in Kaggle Settings → Accelerator.")

Python : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA   : 12.8
GPUs   : 2
  GPU 0: Tesla T4 | 14912 MB
  GPU 1: Tesla T4 | 14912 MB


In [3]:
SHARED_CODE = '''
import os, time, json, subprocess
import torch, torch.nn as nn
import torchvision, torchvision.transforms as T
from torchvision.models import resnet50

# ── Hyperparameters ────────────────────────────────────────────
BATCH_SIZE  = 64
EPOCHS      = 5
LR          = 0.01
NUM_WORKERS = 2
DATA_DIR    = "/kaggle/working/cifar10"
RESULTS_PATH = "/kaggle/working/benchmark_results.json"

def get_loader(rank=0, world_size=1, distributed=False):
    tfm = T.Compose([
        T.Resize(224),
        T.ToTensor(),
        T.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)),
    ])
    ds = torchvision.datasets.CIFAR10(
        root=DATA_DIR, train=True, download=True, transform=tfm)
    sampler = (
        torch.utils.data.distributed.DistributedSampler(
            ds, num_replicas=world_size, rank=rank, shuffle=True)
        if distributed else None
    )
    loader = torch.utils.data.DataLoader(
        ds, batch_size=BATCH_SIZE,
        shuffle=(sampler is None), sampler=sampler,
        num_workers=NUM_WORKERS, pin_memory=True)
    return loader, sampler

def build_model():
    m = resnet50(weights=None)
    m.fc = nn.Linear(m.fc.in_features, 10)
    return m

def peak_mem_mb(device_id=0):
    return torch.cuda.max_memory_allocated(device_id) / 1024**2

def gpu_power_watts(device_ids):
    ids = ",".join(str(i) for i in device_ids)
    out = subprocess.check_output(
        ["nvidia-smi", f"--id={ids}",
         "--query-gpu=power.draw",
         "--format=csv,noheader,nounits"], text=True
    ).strip().split("\\n")
    return sum(float(w) for w in out if w.strip())

def save_result(framework, data):
    try:
        with open(RESULTS_PATH) as f: all_r = json.load(f)
    except FileNotFoundError:
        all_r = {}
    all_r[framework] = data
    with open(RESULTS_PATH, "w") as f: json.dump(all_r, f, indent=2)
    print(f"[{framework}] saved → {RESULTS_PATH}")
'''

with open("/kaggle/working/shared_utils.py", "w") as f:
    f.write(SHARED_CODE)

exec(compile(SHARED_CODE, "shared_utils", "exec"))
print("Shared utilities ready.")

Shared utilities ready.


In [4]:
import subprocess, os, json

BASELINE_SCRIPT = '''
import sys, time, gc
sys.path.insert(0, "/kaggle/working/")
from shared_utils import *
import torch, torch.nn as nn
from torch.amp import autocast, GradScaler

device = torch.device("cuda:0")
loader, _ = get_loader()
model     = build_model().to(device)
opt       = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9)
criterion = nn.CrossEntropyLoss()
scaler    = GradScaler("cuda")

epoch_times, powers = [], []
for epoch in range(EPOCHS):
    model.train()
    torch.cuda.reset_peak_memory_stats()
    t0, n = time.time(), 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        opt.zero_grad()
        with autocast("cuda"):
            loss = criterion(model(imgs), labels)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        n += imgs.size(0)
    dt = time.time() - t0
    epoch_times.append(dt)
    powers.append(gpu_power_watts([0]))
    print(f"  [Baseline] Epoch {epoch+1}/{EPOCHS} | {dt:.1f}s | {n/dt:.0f} samp/s", flush=True)

avg_t = sum(epoch_times) / EPOCHS
avg_p = sum(powers) / EPOCHS
save_result("baseline_1gpu", {
    "throughput_samples_per_sec": round(len(loader.dataset) / avg_t, 2),
    "avg_epoch_time_sec":         round(avg_t, 2),
    "peak_memory_mb":             round(peak_mem_mb(), 2),
    "energy_per_epoch_wh":        round(avg_p * avg_t / 3600, 4),
    "scaling_efficiency":         100.0,
    "num_gpus":                   1,
})
print(f"BASELINE_TIME={avg_t:.4f}")
'''

with open("/kaggle/working/run_baseline.py", "w") as f:
    f.write(BASELINE_SCRIPT)

r = subprocess.run(["python3", "/kaggle/working/run_baseline.py"], timeout=3600)

# Parse baseline_time from saved results
with open("benchmark_results.json") as f:
    baseline_time = json.load(f)["baseline_1gpu"]["avg_epoch_time_sec"]
print(f"\nBaseline T1 = {baseline_time:.2f}s")

100%|██████████| 170M/170M [00:06<00:00, 28.4MB/s] 


  [Baseline] Epoch 1/5 | 190.9s | 262 samp/s
  [Baseline] Epoch 2/5 | 197.9s | 253 samp/s
  [Baseline] Epoch 3/5 | 198.2s | 252 samp/s
  [Baseline] Epoch 4/5 | 198.6s | 252 samp/s
  [Baseline] Epoch 5/5 | 198.4s | 252 samp/s
[baseline_1gpu] saved → /kaggle/working/benchmark_results.json
BASELINE_TIME=196.7862

Baseline T1 = 196.79s


In [5]:
DDP_SCRIPT = r'''
import os, sys, time
sys.path.insert(0, "/kaggle/working/")
from shared_utils import *
import torch, torch.nn as nn, torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
dist.init_process_group(backend="nccl")
rank, ws = dist.get_rank(), dist.get_world_size()
device = torch.device(f"cuda:{rank}")
torch.cuda.set_device(device)

loader, sampler = get_loader(rank, ws, distributed=True)
model = DDP(build_model().to(device), device_ids=[rank])
opt   = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9)
crit  = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')

epoch_times, powers = [], []
for ep in range(EPOCHS):
    sampler.set_epoch(ep)
    model.train()
    torch.cuda.reset_peak_memory_stats(device)
    t0, n = time.time(), 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        opt.zero_grad()
        with torch.amp.autocast('cuda'):
            loss = crit(model(imgs), labels)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        n += imgs.size(0)
    dt = time.time() - t0
    epoch_times.append(dt)
    if rank == 0:
        powers.append(gpu_power_watts([0,1]))
        print(f"  [DDP] Epoch {ep+1}/{EPOCHS} | {dt:.1f}s | {n*ws/dt:.0f} samp/s", flush=True)

if rank == 0:
    avg_t = sum(epoch_times)/EPOCHS
    avg_p = sum(powers)/EPOCHS
    T1    = float(os.environ.get("BASELINE_TIME", avg_t*2))
    save_result("DDP", {
        "throughput_samples_per_sec": round(len(loader.dataset)*ws/avg_t, 2),
        "avg_epoch_time_sec":         round(avg_t, 2),
        "peak_memory_mb":             round(peak_mem_mb(0), 2),
        "energy_per_epoch_wh":        round(avg_p*avg_t/3600, 4),
        "scaling_efficiency":         round(T1/(avg_t*ws)*100, 2),
        "num_gpus":                   ws,
    })
dist.destroy_process_group()
'''
with open("/kaggle/working/run_ddp.py", "w") as f: f.write(DDP_SCRIPT)
print("DDP script written.")

DDP script written.


In [6]:
import subprocess, os, sys
env = {**os.environ, "BASELINE_TIME": str(196.79)}
print("Running DDP (2 GPUs)...")
r = subprocess.run(
    ["torchrun", "--nproc_per_node=2", "--master_port=29500", "/kaggle/working/run_ddp.py"],
    env=env, timeout=3600)
if r.returncode != 0:
    print(f"DDP failed with exit code {r.returncode}")

Running DDP (2 GPUs)...


W0608 10:53:52.583000 183 torch/distributed/run.py:852] 
W0608 10:53:52.583000 183 torch/distributed/run.py:852] *****************************************
W0608 10:53:52.583000 183 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0608 10:53:52.583000 183 torch/distributed/run.py:852] *****************************************


  [DDP] Epoch 1/5 | 99.8s | 501 samp/s
  [DDP] Epoch 2/5 | 102.0s | 490 samp/s
  [DDP] Epoch 3/5 | 102.0s | 490 samp/s
  [DDP] Epoch 4/5 | 102.4s | 488 samp/s
  [DDP] Epoch 5/5 | 102.3s | 489 samp/s
[DDP] saved → /kaggle/working/benchmark_results.json


In [7]:
HVD_SCRIPT = r'''
import os, sys, time
sys.path.insert(0, "/kaggle/working/")
from shared_utils import *
import torch, torch.nn as nn, torch.distributed as dist

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
dist.init_process_group(backend="nccl")
rank, ws = dist.get_rank(), dist.get_world_size()
device = torch.device(f"cuda:{rank}")
torch.cuda.set_device(device)

loader, sampler = get_loader(rank, ws, distributed=True)
model = build_model().to(device)

# Horovod broadcasts initial parameters from rank 0
for p in model.parameters():
    dist.broadcast(p.data, src=0)

# Horovod scales LR by world_size (linear scaling rule)
opt  = torch.optim.SGD(model.parameters(), lr=LR * ws, momentum=0.9)
crit = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')

def horovod_allreduce_gradients(model, ws):
    """Ring-allreduce: average gradients across all workers."""
    for p in model.parameters():
        if p.grad is not None:
            # all_reduce sums across workers; divide by ws to get mean
            dist.all_reduce(p.grad.data, op=dist.ReduceOp.SUM)
            p.grad.data /= ws

epoch_times, powers = [], []
for ep in range(EPOCHS):
    sampler.set_epoch(ep)
    model.train()
    torch.cuda.reset_peak_memory_stats(device)
    t0, n = time.time(), 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        opt.zero_grad()
        with torch.amp.autocast('cuda'):
            loss = crit(model(imgs), labels)
        scaler.scale(loss).backward()
        horovod_allreduce_gradients(model, ws)   # <- Horovod's step
        scaler.step(opt)
        scaler.update()
        n += imgs.size(0)
    dt = time.time() - t0
    epoch_times.append(dt)
    if rank == 0:
        powers.append(gpu_power_watts([0,1]))
        print(f"  [Horovod-style] Epoch {ep+1}/{EPOCHS} | {dt:.1f}s | {n*ws/dt:.0f} samp/s", flush=True)

if rank == 0:
    avg_t = sum(epoch_times)/EPOCHS
    avg_p = sum(powers)/EPOCHS
    T1    = float(os.environ.get("BASELINE_TIME", avg_t*2))
    save_result("Horovod_style", {
        "throughput_samples_per_sec": round(len(loader.dataset)*ws/avg_t, 2),
        "avg_epoch_time_sec":         round(avg_t, 2),
        "peak_memory_mb":             round(peak_mem_mb(0), 2),
        "energy_per_epoch_wh":        round(avg_p*avg_t/3600, 4),
        "scaling_efficiency":         round(T1/(avg_t*ws)*100, 2),
        "num_gpus":                   ws,
    })
dist.destroy_process_group()
'''
with open("/kaggle/working/run_horovod.py", "w") as f: f.write(HVD_SCRIPT)
print("Horovod-style script written.")

Horovod-style script written.


In [8]:
import subprocess, os
env = {**os.environ, "BASELINE_TIME": str(219.3208)}
print("Running Horovod-style (2 GPUs)...")
r = subprocess.run(
    ["torchrun", "--nproc_per_node=2", "--master_port=29501", "/kaggle/working/run_horovod.py"],
    env=env, timeout=3600)
if r.returncode != 0:
    print(f"Horovod-style failed with exit code {r.returncode}")

Running Horovod-style (2 GPUs)...


W0608 11:15:34.112000 329 torch/distributed/run.py:852] 
W0608 11:15:34.112000 329 torch/distributed/run.py:852] *****************************************
W0608 11:15:34.112000 329 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0608 11:15:34.112000 329 torch/distributed/run.py:852] *****************************************


  [Horovod-style] Epoch 1/5 | 99.2s | 504 samp/s
  [Horovod-style] Epoch 2/5 | 104.6s | 478 samp/s
  [Horovod-style] Epoch 3/5 | 105.6s | 474 samp/s
  [Horovod-style] Epoch 4/5 | 105.4s | 475 samp/s
  [Horovod-style] Epoch 5/5 | 105.5s | 474 samp/s
[Horovod_style] saved → /kaggle/working/benchmark_results.json


In [9]:
FSDP_SCRIPT = r'''
import os, sys, time, functools
sys.path.insert(0, "/kaggle/working/")
from shared_utils import *
import torch, torch.nn as nn, torch.distributed as dist
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP, ShardingStrategy
from torch.distributed.fsdp.wrap import size_based_auto_wrap_policy

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
dist.init_process_group(backend="nccl")
rank, ws = dist.get_rank(), dist.get_world_size()
device = torch.device(f"cuda:{rank}")
torch.cuda.set_device(device)

loader, sampler = get_loader(rank, ws, distributed=True)

# SHARD_GRAD_OP = ZeRO-1/2: shard optimizer states & gradients, replicate params
model = FSDP(
    build_model().to(device),
    sharding_strategy=ShardingStrategy.SHARD_GRAD_OP,
    auto_wrap_policy=functools.partial(size_based_auto_wrap_policy, min_num_params=1_000_000),
    device_id=rank,
)
opt  = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9)
scaler = torch.amp.GradScaler('cuda')
crit = nn.CrossEntropyLoss()

epoch_times, powers = [], []
for ep in range(EPOCHS):
    sampler.set_epoch(ep)
    model.train()
    torch.cuda.reset_peak_memory_stats(device)
    t0, n = time.time(), 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        opt.zero_grad()
        with torch.amp.autocast('cuda'):
            loss = crit(model(imgs), labels)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        n += imgs.size(0)
    dt = time.time() - t0
    epoch_times.append(dt)
    if rank == 0:
        powers.append(gpu_power_watts([0,1]))
        print(f"  [FSDP] Epoch {ep+1}/{EPOCHS} | {dt:.1f}s | {n*ws/dt:.0f} samp/s", flush=True)

if rank == 0:
    avg_t = sum(epoch_times)/EPOCHS
    avg_p = sum(powers)/EPOCHS
    T1    = float(os.environ.get("BASELINE_TIME", avg_t*2))
    save_result("FSDP_Zero1", {
        "throughput_samples_per_sec": round(len(loader.dataset)*ws/avg_t, 2),
        "avg_epoch_time_sec":         round(avg_t, 2),
        "peak_memory_mb":             round(peak_mem_mb(0), 2),
        "energy_per_epoch_wh":        round(avg_p*avg_t/3600, 4),
        "scaling_efficiency":         round(T1/(avg_t*ws)*100, 2),
        "num_gpus":                   ws,
    })
dist.destroy_process_group()
'''
with open("/kaggle/working/run_fsdp.py", "w") as f: f.write(FSDP_SCRIPT)
print("FSDP script written.")

FSDP script written.


In [10]:
import subprocess, os
env = {**os.environ, "BASELINE_TIME": str(196.79)}
print("Running FSDP-Zero1 (2 GPUs)...")
r = subprocess.run(
    ["torchrun", "--nproc_per_node=2", "--master_port=29502", "/kaggle/working/run_fsdp.py"],
    env=env, timeout=3600)
if r.returncode != 0:
    print(f"FSDP failed with exit code {r.returncode}")

Running FSDP-Zero1 (2 GPUs)...


W0608 11:40:42.101000 475 torch/distributed/run.py:852] 
W0608 11:40:42.101000 475 torch/distributed/run.py:852] *****************************************
W0608 11:40:42.101000 475 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0608 11:40:42.101000 475 torch/distributed/run.py:852] *****************************************


  [FSDP] Epoch 1/5 | 93.4s | 535 samp/s
  [FSDP] Epoch 2/5 | 95.4s | 524 samp/s
  [FSDP] Epoch 3/5 | 96.0s | 521 samp/s
  [FSDP] Epoch 4/5 | 95.9s | 521 samp/s
  [FSDP] Epoch 5/5 | 96.0s | 521 samp/s
[FSDP_Zero1] saved → /kaggle/working/benchmark_results.json


In [2]:
import json, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

with open("/kaggle/working/benchmark_results.json") as f:
    raw = json.load(f)

df_all = pd.DataFrame(raw).T
df = df_all.loc[[k for k in raw if k != "baseline_1gpu"]].copy()
df.index = ["DDP", "Horovod-style", "FSDP-Zero1"]

cols = {
    "throughput_samples_per_sec": "Throughput (samp/s)",
    "scaling_efficiency":         "Scaling Effic. (%)",
    "energy_per_epoch_wh":        "Energy/Epoch (Wh)",
    "peak_memory_mb":             "Peak Mem (MB)",
}
display_df = df[list(cols.keys())].rename(columns=cols).astype(float)
print("\n" + "="*62)
print("  BENCHMARK RESULTS (2× T4 GPU, ResNet-50, CIFAR-10)")
print("="*62)
print(display_df.to_string(float_format="{:.2f}".format))
print("="*62)

print("\n📊 KEY FINDINGS:")
print(f"  RQ1 Highest Throughput  → {display_df['Throughput (samp/s)'].idxmax()}")
print(f"  RQ2 Best Scaling Effic. → {display_df['Scaling Effic. (%)'].idxmax()}")
print(f"  RQ3 Lowest Energy/Epoch → {display_df['Energy/Epoch (Wh)'].idxmin()}")
print(f"  RQ4 Lowest Peak Memory  → {display_df['Peak Mem (MB)'].idxmin()}")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/benchmark_results.json'

In [1]:
COLORS   = ["#1565C0", "#BF360C", "#2E7D32"]
LABELS   = display_df.index.tolist()
metrics  = list(cols.values())
higher_better = {"Throughput (samp/s)": True, "Scaling Effic. (%)": True,
                 "Energy/Epoch (Wh)": False, "Peak Mem (MB)": False}

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
rqs = ["RQ1", "RQ2", "RQ3", "RQ4"]

for ax, metric, rq in zip(axes.flat, metrics, rqs):
    vals = display_df[metric].values.astype(float)
    bars = ax.bar(LABELS, vals, color=COLORS, width=0.5,
                  edgecolor="white", linewidth=1.5)
    best = int(np.argmax(vals) if higher_better[metric] else np.argmin(vals))
    bars[best].set_edgecolor("#FFD600")
    bars[best].set_linewidth(3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() * 1.015,
                f"{v:.1f}", ha="center", fontsize=9, fontweight="bold")
    ax.set_title(f"{metric}  [{rq}]", fontsize=11, fontweight="bold", pad=8)
    ax.set_ylabel(metric, fontsize=9)
    ax.yaxis.grid(True, linestyle="--", alpha=0.4)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="x", labelsize=10)

fig.suptitle(
    "Distributed DL Benchmark: DDP vs Horovod-style vs FSDP-Zero1\n"
    "ResNet-50 · CIFAR-10 · Kaggle T4 × 2  (gold border = winner)",
    fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("benchmark_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to /tmp/benchmark_results.png")

NameError: name 'display_df' is not defined